In [ ]:
!pip install fastf1

import os
import fastf1
import pandas as pd
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')


os.makedirs('cache', exist_ok=True)
fastf1.Cache.enable_cache('cache')

print("Libraries imported successfully!")

import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)


Libraries imported successfully!


Driver map to get the name from the number

In [19]:
DRIVER_MAP_2024 = {
    1:  ("VER", "Max Verstappen"),
    11: ("PER", "Sergio Pérez"),
    16: ("LEC", "Charles Leclerc"),
    55: ("SAI", "Carlos Sainz"),
    63: ("RUS", "George Russell"),
    44: ("HAM", "Lewis Hamilton"),
    4:  ("NOR", "Lando Norris"),
    81: ("PIA", "Oscar Piastri"),
    14: ("ALO", "Fernando Alonso"),
    18: ("STR", "Lance Stroll"),
    23: ("ALB", "Alex Albon"),
    2:  ("SAR", "Logan Sargeant"),
    20: ("MAG", "Kevin Magnussen"),
    27: ("HUL", "Nico Hülkenberg"),
    24: ("ZHO", "Guanyu Zhou"),
    77: ("BOT", "Valtteri Bottas"),
    10: ("GAS", "Pierre Gasly"),
    31: ("OCO", "Esteban Ocon"),
    22: ("TSU", "Yuki Tsunoda"),
    3:  ("RIC", "Daniel Ricciardo"),
}

def get_driver_info(session, driver):

    try:
        drv = session.get_driver(driver)
        return drv['Abbreviation'], drv['FullName']
    except:
        return "UNK", "Unknown"



In [ ]:
# Configuration
YEAR = 2024

N_RACES = 24  # Number of past races to analyze

# R for Race 
# Q for Qualifying 
# Sprint for Sprint
SESSION_TYPE = 'R'  


SPECIFIC_RACES = []
print(f"Configuration set: Analyzing {N_RACES} races from {YEAR}")

Configuration set: Analyzing 24 races from 2024


Get the race details for each race

In [8]:
def get_race_list(year, n_races, specific_races=None):

    # getting the list of races
    schedule = fastf1.get_event_schedule(year)


    # Get last n completed races
    # Filter out future races and testing
    completed = schedule[schedule['EventFormat'] != 'testing']
    # You might want to add date filtering here for completed races
    races = completed.tail(n_races)
    
    return races[['EventName', 'Location', 'RoundNumber']]

# Get the race list
race_list = get_race_list(YEAR, N_RACES, SPECIFIC_RACES)
print(f"\nRaces to analyze:")
print(race_list)


Races to analyze:
                    EventName           Location  RoundNumber
1          Bahrain Grand Prix             Sakhir            1
2    Saudi Arabian Grand Prix             Jeddah            2
3       Australian Grand Prix          Melbourne            3
4         Japanese Grand Prix             Suzuka            4
5          Chinese Grand Prix           Shanghai            5
6            Miami Grand Prix              Miami            6
7   Emilia Romagna Grand Prix              Imola            7
8           Monaco Grand Prix             Monaco            8
9         Canadian Grand Prix           Montréal            9
10         Spanish Grand Prix          Barcelona           10
11        Austrian Grand Prix          Spielberg           11
12         British Grand Prix        Silverstone           12
13       Hungarian Grand Prix           Budapest           13
14         Belgian Grand Prix  Spa-Francorchamps           14
15           Dutch Grand Prix          Zandvoort   

Get the sector times of each driver 

In [9]:
def get_driver_best_sectors(laps, driver):
    
    # get all of their race laps
    driver_laps = laps.pick_drivers(driver)
    
    # Filter for valid laps (laps where the driver has not breached track conditions)
    valid_laps = driver_laps.pick_accurate().pick_not_deleted()
    
    # sort them by laptime so we can use index 0 for fastest and we can get the average 
    valid_laps = valid_laps.sort_values('LapTime')
    
    if len(valid_laps) == 0:
        return None
    
    # Get best times
    best_sectors = {
        'Sector1': valid_laps['Sector1Time'].min(),
        'Sector2': valid_laps['Sector2Time'].min(),
        'Sector3': valid_laps['Sector3Time'].min()
    }
    
    return best_sectors

Overall fastest per sector

In [10]:
def get_fastest_per_sector(year, race_name, session_type='R'):
    session = fastf1.get_session(year, race_name, session_type)
    session.load()

    results = {'race': race_name}

    for sec in [1, 2, 3]:
        sec_name = f"Sector{sec}" # sector 1, 2 or 3
        fastest_driver, fastest_time = None, None

        
        for d in session.drivers:
            best = get_driver_best_sectors(session.laps, d)
            if not best or pd.isna(best[sec_name]):
                continue
            

            t = best[sec_name]
            if fastest_time is None or t < fastest_time:
                fastest_time, fastest_driver = t, d

        
        results[sec_name] = {
            'driver': fastest_driver,
            'time': fastest_time
        }

    return results


In [ ]:
# Store results for all races
all_race_results = []

for idx, race in race_list.iterrows():
    race_name = race['EventName']
    result = get_fastest_per_sector(YEAR, race_name, SESSION_TYPE)
    
    if result:
        all_race_results.append(result)

print(f"\n Completed analysis for {len(all_race_results)} races")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca


✓ Completed analysis for 24 races


In [23]:
def display_sector_results(results):
    print("\n" + "="*70)
    print("FASTEST DRIVER PER SECTOR - RACE BY RACE")
    print("="*70)

    for result in results:
        print(f"\n {result['race']}")
        print("-" * 50)

        for sector in ['Sector1', 'Sector2', 'Sector3']:
            driver_raw = result[sector]['driver']
            time = result[sector]['time']

            # Normalize driver number
            try:
                driver_num = int(driver_raw)
            except:
                driver_num = None

            if driver_num and pd.notna(time):
                code, name = DRIVER_MAP_2024.get(driver_num, ("Unknown", "Unknown"))
                time_str = str(time).split()[-1]
                print(f"  {sector}: {name} ({code}) - {time_str}")
            else:
                print(f"  {sector}: No data")


# Display results
display_sector_results(all_race_results)


FASTEST DRIVER PER SECTOR - RACE BY RACE

 Bahrain Grand Prix
--------------------------------------------------
  Sector1: Max Verstappen (VER) - 00:00:29.741000
  Sector2: Max Verstappen (VER) - 00:00:39.916000
  Sector3: Max Verstappen (VER) - 00:00:22.951000

 Saudi Arabian Grand Prix
--------------------------------------------------
  Sector1: Lando Norris (NOR) - 00:00:33.575000
  Sector2: Charles Leclerc (LEC) - 00:00:28.668000
  Sector3: Lewis Hamilton (HAM) - 00:00:28.873000

 Australian Grand Prix
--------------------------------------------------
  Sector1: Lando Norris (NOR) - 00:00:27.633000
  Sector2: Kevin Magnussen (MAG) - 00:00:17.488000
  Sector3: Charles Leclerc (LEC) - 00:00:34.197000

 Japanese Grand Prix
--------------------------------------------------
  Sector1: Carlos Sainz (SAI) - 00:00:33.310000
  Sector2: George Russell (RUS) - 00:00:42.024000
  Sector3: Max Verstappen (VER) - 00:00:18.161000

 Chinese Grand Prix
------------------------------------------

In [24]:
def get_team_lineups(session):

    results = session.results
    
    team_lineups = {}
    for idx, driver in results.iterrows():
        team = driver['TeamName']
        driver_num = driver['DriverNumber']
        
        if team not in team_lineups:
            team_lineups[team] = []
        
        if driver_num not in team_lineups[team]:
            team_lineups[team].append(driver_num)
    
    # Only keep teams with exactly 2 drivers
    team_lineups = {team: drivers for team, drivers in team_lineups.items() 
                    if len(drivers) == 2}
    
    return team_lineups

In [25]:
def analyze_team_head_to_head(year, race_name, session_type='R'):
    """
    Compare teammates sector by sector — with driver names + codes.
    """
    print(f"\nLoading {race_name} for team comparison...")

    try:
        # Load session
        session = fastf1.get_session(year, race_name, session_type)
        session.load()

        # Get team lineups
        team_lineups = get_team_lineups(session)

        race_comparison = {
            'race': race_name,
            'teams': {}
        }

        for team, drivers in team_lineups.items():
            if len(drivers) != 2:
                continue

            driver1, driver2 = drivers[0], drivers[1]

            # Sector bests
            d1_sectors = get_driver_best_sectors(session.laps, driver1)
            d2_sectors = get_driver_best_sectors(session.laps, driver2)

            if d1_sectors is None or d2_sectors is None:
                continue

            # Driver info
            d1_code, d1_name = get_driver_info(session, driver1)
            d2_code, d2_name = get_driver_info(session, driver2)

            team_result = {
                'driver1': {'num': driver1, 'code': d1_code, 'name': d1_name},
                'driver2': {'num': driver2, 'code': d2_code, 'name': d2_name},
                'sectors': {}
            }

            for sector in ['Sector1', 'Sector2', 'Sector3']:
                d1_time = d1_sectors[sector]
                d2_time = d2_sectors[sector]

                if pd.notna(d1_time) and pd.notna(d2_time):
                    winner = 'driver1' if d1_time < d2_time else 'driver2'
                    delta = abs(d1_time - d2_time)

                    team_result['sectors'][sector] = {
                        'winner': winner,
                        'winner_code': team_result[winner]['code'],
                        'winner_name': team_result[winner]['name'],
                        'd1_time': d1_time,
                        'd2_time': d2_time,
                        'delta': delta
                    }

            race_comparison['teams'][team] = team_result

        return race_comparison

    except Exception as e:
        print(f"Error loading {race_name}: {e}")
        return None


In [26]:
# Store team comparisons for all races
all_team_comparisons = []

for idx, race in race_list.iterrows():
    race_name = race['EventName']
    comparison = analyze_team_head_to_head(YEAR, race_name, SESSION_TYPE)
    
    if comparison:
        all_team_comparisons.append(comparison)

print(f"\n✓ Completed team comparison for {len(all_team_comparisons)} races")


Loading Bahrain Grand Prix for team comparison...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix


Loading Saudi Arabian Grand Prix for team comparison...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...



Loading Australian Grand Prix for team comparison...


...


Loading Qatar Grand Prix for team comparison...


core        WARNING 	Fixed incorrect tyre stint information for driver '43'
core        WARNING 	Fixed incorrect tyre stint information for driver '31'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '10', '55', '14', '24', '20', '4', '77', '44', '22', '30', '23', '27', '11', '18', '43', '31']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req         


Loading Abu Dhabi Grand Prix for team comparison...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']



✓ Completed team comparison for 24 races


In [31]:
def display_team_comparison_by_race(comparisons):
    """
    Simplified printout:
    Race Name
    Driver1 vs Driver2
    Section X - Winner - Delta
    """
    
    for comparison in comparisons:
        print(comparison['race'])
        
        # Get team (only one per comparison)
        team_data = list(comparison['teams'].values())[0]
        
        d1 = team_data['driver1']
        d2 = team_data['driver2']
        
        print(f"{d1['name']} ({d1['code']}) vs {d2['name']} ({d2['code']})")
        
        for sector, sector_data in team_data['sectors'].items():
            winner_key = sector_data['winner']
            
            if winner_key == "driver1":
                winner_name = d1['name']
            else:
                winner_name = d2['name']
            
            delta = str(sector_data['delta']).split()[-1]
            print(f"{sector} - {winner_name} - {delta}")
        
        print()  # Blank line between races


# Display results
display_team_comparison_by_race(all_team_comparisons)

Bahrain Grand Prix
Max Verstappen (VER) vs Sergio Perez (PER)
Sector1 - Max Verstappen - 00:00:00.322000
Sector2 - Max Verstappen - 00:00:00.856000
Sector3 - Max Verstappen - 00:00:00.200000

Saudi Arabian Grand Prix
Max Verstappen (VER) vs Sergio Perez (PER)
Sector1 - Max Verstappen - 00:00:00.038000
Sector2 - Max Verstappen - 00:00:00.228000
Sector3 - Max Verstappen - 00:00:00.076000

Australian Grand Prix
Carlos Sainz (SAI) vs Charles Leclerc (LEC)
Sector1 - Charles Leclerc - 00:00:00.046000
Sector2 - Charles Leclerc - 00:00:00.249000
Sector3 - Charles Leclerc - 00:00:00.015000

Japanese Grand Prix
Max Verstappen (VER) vs Sergio Perez (PER)
Sector1 - Max Verstappen - 00:00:00.029000
Sector2 - Max Verstappen - 00:00:00.122000
Sector3 - Max Verstappen - 00:00:00.109000

Chinese Grand Prix
Max Verstappen (VER) vs Sergio Perez (PER)
Sector1 - Max Verstappen - 00:00:00.225000
Sector2 - Max Verstappen - 00:00:00.534000
Sector3 - Max Verstappen - 00:00:00.287000

Miami Grand Prix
Lando Nor

In [ ]:
def aggregate_team_stats(comparisons):
    """
    Aggregate team performance across all races
    """
    team_stats = {}

    for comparison in comparisons:
        for team, data in comparison['teams'].items():

            d1 = data['driver1']
            d2 = data['driver2']

            if team not in team_stats:
                team_stats[team] = {
                    'driver1': d1,
                    'driver2': d2,
                    'Sector1': {'d1_wins': 0, 'd2_wins': 0, 'deltas': []},
                    'Sector2': {'d1_wins': 0, 'd2_wins': 0, 'deltas': []},
                    'Sector3': {'d1_wins': 0, 'd2_wins': 0, 'deltas': []}
                }

            for sector, sector_data in data['sectors'].items():
                # winner stored as 'driver1'/'driver2' and winner_code stored as code
                winner_key = sector_data.get('winner')            # 'driver1' or 'driver2'
                winner_code = sector_data.get('winner_code')

                # normalize winner_code (fallback to driver key lookup)
                if winner_code:
                    w_code = str(winner_code).strip().upper()
                else:
                    if winner_key == 'driver1':
                        w_code = str(d1['code']).strip().upper()
                    elif winner_key == 'driver2':
                        w_code = str(d2['code']).strip().upper()
                    else:
                        w_code = str(winner_key).strip().upper()

                d1_code = str(d1['code']).strip().upper()
                d2_code = str(d2['code']).strip().upper()
                delta = sector_data.get('delta')

                if w_code == d1_code:
                    team_stats[team][sector]['d1_wins'] += 1
                elif w_code == d2_code:
                    team_stats[team][sector]['d2_wins'] += 1
                else:
                    print("WARNING: Winner does not match either driver for", team, winner_key, d1_code, d2_code)

                if delta is not None:
                    team_stats[team][sector]['deltas'].append(delta)

    return team_stats


In [55]:
def display_aggregated_stats(team_stats, n_races):
    """
    Display total head-to-head results:
    Team
    Driver1 vs Driver2
    Driver1 beats Driver2 X–Y
    """
    print(f"\nAGGREGATED TEAM STATISTICS (Last {n_races} Races)\n")
    
    for team, stats in team_stats.items():
        d1 = stats["driver1"]
        d2 = stats["driver2"]

        d1_name = d1["name"]
        d2_name = d2["name"]
        
        total_d1 = sum(stats[sector]["d1_wins"] for sector in ["Sector1", "Sector2", "Sector3"])
        total_d2 = sum(stats[sector]["d2_wins"] for sector in ["Sector1", "Sector2", "Sector3"])
        
        print(team)
        print(f"{d1_name} vs {d2_name}")
        
        if total_d1 > total_d2:
            print(f"{d1_name} beats {d2_name} {total_d1}-{total_d2} at a {round((total_d1 / (total_d2 + total_d1)) * 100, 2)}% win rate")
        elif total_d2 > total_d1:
            print(f"{d2_name} beats {d1_name} {total_d2}-{total_d1} at a {round((total_d2 / (total_d1 + total_d2) * 100), 2)}% win rate")
        else:
            print(f"{d1_name} and {d2_name} tied {total_d1}-{total_d2} at a {round((total_d1 / (total_d2 + total_d1)) * 100, 2)}% win rate")
        
        print()



# Display aggregated results
team_stats = aggregate_team_stats(all_team_comparisons)
display_aggregated_stats(team_stats, N_RACES)


AGGREGATED TEAM STATISTICS (Last 24 Races)

Red Bull Racing
Max Verstappen vs Sergio Perez
Max Verstappen beats Sergio Perez 50-16 at a 75.76% win rate

Ferrari
Carlos Sainz vs Charles Leclerc
Carlos Sainz beats Charles Leclerc 39-33 at a 54.17% win rate

Mercedes
George Russell vs Lewis Hamilton
George Russell beats Lewis Hamilton 41-28 at a 59.42% win rate

McLaren
Lando Norris vs Oscar Piastri
Lando Norris beats Oscar Piastri 38-34 at a 52.78% win rate

Aston Martin
Fernando Alonso vs Lance Stroll
Fernando Alonso beats Lance Stroll 41-28 at a 59.42% win rate

Kick Sauber
Guanyu Zhou vs Valtteri Bottas
Guanyu Zhou beats Valtteri Bottas 45-27 at a 62.5% win rate

Haas F1 Team
Kevin Magnussen vs Nico Hulkenberg
Kevin Magnussen beats Nico Hulkenberg 39-30 at a 56.52% win rate

RB
Daniel Ricciardo vs Yuki Tsunoda
Daniel Ricciardo beats Yuki Tsunoda 44-22 at a 66.67% win rate

Williams
Alexander Albon vs Logan Sargeant
Alexander Albon beats Logan Sargeant 37-20 at a 64.91% win rate

Alpi